# 03 · Entrenamiento del Random Forest

Búsqueda de hiperparámetros con validación cruzada estratificada, optimizando **F1**.

In [ ]:
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

train = pd.read_csv("../data/train_balanceado.csv")
X_tr, y_tr = train.drop(columns="en_riesgo"), train["en_riesgo"]

In [ ]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid={
        "n_estimators": [100, 200, 400],
        "max_depth": [6, 8, 12, None],
        "min_samples_leaf": [1, 3, 5],
    },
    scoring="f1",
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    n_jobs=-1,
)
grid.fit(X_tr, y_tr)
print(grid.best_params_, round(grid.best_score_, 3))

In [ ]:
modelo = grid.best_estimator_
pd.Series(modelo.feature_importances_, index=X_tr.columns).sort_values().plot.barh(title="Importancia de variables")

## Exportar
⚠️ El orden de columnas debe coincidir con `ml-service/app/features.py`. Si el VIF eliminó variables, actualiza ese archivo.

In [ ]:
joblib.dump(modelo, "../../ml-service/model/model.joblib")